# Task E — Written justification and supporting calculations

CDB601220 · Week 3. Work through the cells from top to bottom. Every dataset is loaded from the **data/** folder beside this notebook. All data are synthetic, supplied with the assignment. Charts and written answers stay inside the notebook; no output files are created.

## 1. Import libraries and locate the data folder

In [1]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display

DATA = Path('data')
assert DATA.is_dir(), 'Open this notebook inside task3, beside the data folder.'
sns.set_theme(style='whitegrid', palette=['#0072B2', '#D55E00', '#009E73'])
plt.ioff()
BLUE, ORANGE, GREEN = '#0072B2', '#D55E00', '#009E73'
SOURCE = 'Source: synthetic CDB601220 Week 3 assignment data.'

## 2. Load the data to verify the numbers used below

In [2]:
raw = pd.read_csv(DATA / 'vitals_10k.csv')
print('Original patient rows:', len(raw))

Original patient rows: 10000


### Verify heart-rate exclusions and mean changes

In [3]:
high_hr = raw.heart_rate > 250
zero_hr = raw.heart_rate <= 0
display(pd.DataFrame({
    'stage':['Original','After excluding >250','After also excluding zero'],
    'rows':[len(raw), (~high_hr).sum(), (~(high_hr|zero_hr)).sum()],
    'mean_hr':[raw.heart_rate.mean(), raw.loc[~high_hr,'heart_rate'].mean(),
               raw.loc[~(high_hr|zero_hr),'heart_rate'].mean()]}))
print('High-HR exclusions:', high_hr.sum(), '; zero-HR exclusions:', zero_hr.sum())

,stage,rows,mean_hr
0,Original,10000,78.833100
1,After excluding >250,9971,77.984254
2,After also excluding zero,9963,78.046873


High-HR exclusions: 29 ; zero-HR exclusions: 8


### Verify BMI replacement counts and mean changes

In [4]:
bmi_bad = raw.bmi.isin([0,999])
display(raw.loc[bmi_bad,'bmi'].value_counts())
print('Original BMI mean:', raw.bmi.mean())
print('After sentinel replacement:', raw.loc[~bmi_bad,'bmi'].mean())
print('Valid BMI after HR filtering:', (~bmi_bad & ~(high_hr|zero_hr)).sum())

bmi
999.0    402
0.0       84
Name: count, dtype: int64

Original BMI mean: 66.17703999999999
After sentinel replacement: 27.346268656716422
Valid BMI after HR filtering: 9478


## 3. Question answered by each panel

**Panel 1 — Are systolic values from one distribution?** A histogram reveals two modes. Common-axis position and zero-based count length support comparison.

**Panel 2 — Which wards have longer stays?** Box plots compare medians and quartiles while retaining all outliers. Aligned position is more precise than area. ICU median is 3.47 days versus 0.25 in Day Surgery.

**Panel 3 — What does the strongest measured pair look like?** A scatter shows systolic versus diastolic pressure (r≈0.888), chosen after examining the heatmap. X/Y position shows the joint pattern, while circles/squares duplicate sex colors.

**Panel 4 — Which measurements warrant investigation?** A triangular heatmap screens signed Pearson correlations. A diverging palette centered at zero represents sign and strength; the chart does not establish causation.

## 4. Parameters chosen and why

- **Histogram:** edges 89.5–209.5 mmHg, width 5 mmHg. Checks at widths 2/5/10 retain the modes. Counts start at zero; no smoothing.
- **Box plots:** width 0.6, 1.5-IQR whiskers, all fliers shown with 2-point markers, wards ordered by median. The symlog axis uses base 10, linear threshold 0.1 day and linscale 1, retaining the two rounded zero-day stays while compressing the long tail.
- **Scatter:** all retained records, area 9 pt², alpha 0.12, no jitter or subsampling; color and shape encode sex.
- **Heatmap:** pairwise-complete Pearson r, minimum two observations, two-decimal annotations, RdBu_r, limits −1 to 1, center 0, upper triangle and diagonal masked. Actual pair counts are 9,478–9,963.
- **Variable scope:** derived MAP, IDs, dates, categories, binary readmission and the denominator-sensitive risk ratio are omitted from this continuous-measurement screen.
- **Presentation:** 15×11-inch 2×2 figure, tested blue/orange palette with redundant shapes, units, finding titles, source note and spines removed. Charts are displayed inside the notebook without exporting files.

## 5. Cleaning rules, counts and changes

All 10,000 IDs are unique. Twelve ward spellings become four wards; explicit ISO, day/month/year and day-abbreviated-month-year formats parse all dates. These operations remove no rows.

BMI 0 (84 records) and 999 (402) become missing values: mean BMI changes **66.177 → 27.346 kg/m²**. Patient rows are retained.

Readings above 250 bpm are excluded as the brief’s device faults (**29 records**); mean HR moves **78.833 → 77.984 bpm**. Excluding eight invalid zero-HR records next gives **78.047 bpm** and **9,963 retained patient rows**. No other patient rows are removed. Valid BMI after filtering is 9,478.

Task D shows the same figure with both HR exclusions reversed. These value-based rules would still be applied if they weakened the finding: that is the symmetry test.

## 6. Remaining issues and source

SpO₂ is bounded at 100%; no unbounded KDE is used in the clinical review. Bimodal SBP, skewed glucose and long LOS tails are retained as distributional features. risk_score is not averaged into an overall event rate because its original numerator and exposure are unavailable.

The separate wide LOS data are melted from 16,072 cells; 6,072 structural padding blanks are removed, leaving 10,000 observations verified against the main data. Because that file lacks patient IDs, the final boxes use the cleaned main file in long form.

**Source:** original CDB601220 Week 3 assignment instructions, data dictionary and course toolkit; all ten supplied CSVs are synthetic and are preserved in `data/`. This justification is written as notebook cells, as requested.